# Exercise: image processing

**Duration** ~35 min &nbsp;·&nbsp; **Session** Day 1, Python notebooks

You will build a complete cleanup pipeline on the cell channel: the one that
thresholding handled badly in the last exercise. Take the hints in order if you
get stuck.

**Data**: `data/misc/cells_shaded.tif`, and `data/bbbc020/` fields `1h_1` and `15min_3`.

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt

from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label
from skimage.color import label2rgb

from course import DATA, show

## Task 1: measure the vignetting

Load `data/misc/cells_shaded.tif` and display it. You should be able to *see*
that the middle is brighter than the edges: this is **vignetting**, and it is a
property of the optics, not the sample.

Measure it: compare the mean intensity of a 90x90 patch at the centre of the
image with the mean of four 60x60 patches at the corners. Print the ratio.

<details>
<summary>Hint 1: how do I take a patch at the centre?</summary>

If `h, w = image.shape`, the centre patch is
`image[h//2 - 45:h//2 + 45, w//2 - 45:w//2 + 45]`.
</details>

<details>
<summary>Hint 2: the four corners</summary>

`image[:60, :60]`, `image[:60, -60:]`, `image[-60:, :60]`, `image[-60:, -60:]`.
Take the mean of each, then the mean of those four numbers.
</details>

In [ ]:
shaded = tifffile.imread(DATA / "misc" / "cells_shaded.tif").astype(float)
h, w = shaded.shape

show(shaded, title="cells_shaded.tif - bright in the middle")
plt.show()


def centre_corner_ratio(image):
    """1.0 means perfectly even illumination."""
    # --- your turn ---
    centre = ...    # TODO: mean of the 90x90 centre patch
    corners = ...   # TODO: mean of the four 60x60 corner patches
    return centre / corners


print(f"centre / corner ratio: {centre_corner_ratio(shaded):.2f}")

## Task 2: flatten the illumination

The walkthrough removed a *background* by subtracting it. Vignetting is
different: the optics scale the signal down towards the edges, so the effect is
**multiplicative**, and the correction is a **division**.

Estimate the illumination field by blurring the image very heavily, then divide
the image by it. Re-measure the ratio from task 1 to check that it worked.

<details>
<summary>Hint 1: estimating the field</summary>

`background = gaussian(shaded, sigma=37, preserve_range=True)`. The sigma must be
large enough that no cell survives the blur.
</details>

<details>
<summary>Hint 2: dividing without changing the overall brightness</summary>

`flattened = shaded / background * background.mean()`. Multiplying by the mean
puts the result back into roughly the original intensity range.
</details>

In [ ]:
# --- your turn ---
background = ...   # TODO: heavily blurred copy = the illumination field
flattened = ...    # TODO: divide, then rescale by background.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
show(shaded, title=f"before (ratio {centre_corner_ratio(shaded):.2f})", ax=axes[0])
show(background, title="estimated illumination field", ax=axes[1])
show(flattened, title=f"after (ratio {centre_corner_ratio(flattened):.2f})", ax=axes[2])
plt.show()

The ratio should have dropped from about 4.5 to near 1: the illumination is now
roughly even across the field.

Now look at what it cost. Count objects before and after, with an identical
threshold-and-cleanup on each.

In [ ]:
from scipy.ndimage import binary_fill_holes
from skimage.morphology import remove_small_objects


def count(image):
    mask = image > threshold_otsu(image)
    mask = binary_fill_holes(mask)
    mask = remove_small_objects(mask, max_size=15)
    return label(mask).max()


print("objects before flattening:", count(shaded))
print("objects after  flattening:", count(flattened))

**Your answer:** the count went *up*. That is not a bug: explain what happened.

Hint if you need it: think about what division does to a dim corner where there
was almost no signal, and what happens to the noise that was sitting there.
*(edit this cell)*

This is worth remembering alongside the walkthrough's top-hat: **correcting an
image and improving a segmentation are not the same thing**. The illumination is
genuinely fixed, and the naive count genuinely got worse, because flattening
amplifies noise wherever the signal was weak. You would follow this with
smoothing and a size filter, which is exactly task 4.

## Task 3: which filter for which noise?

Two noisy versions of the same image are made for you below. For each, apply
**both** a Gaussian and a median filter, display all four results, and then say
in the markdown cell which filter suits which noise, and why.

<details>
<summary>Hint</summary>

`gaussian(img, sigma=2, preserve_range=True)` and `median(img, disk(2))`.
Import `median` from `skimage.filters`.
</details>

In [ ]:
rng = np.random.default_rng(1)
base = tifffile.imread(DATA / "bbbc020" / "images" / "1h_1_nuclei.tif")[130:280, 180:330]

grainy = base.astype(float) + rng.normal(0, 20, base.shape)

speckled = base.copy()
spots = rng.random(base.shape)
speckled[spots < 0.03] = 0
speckled[spots > 0.97] = 255

In [ ]:
from skimage.filters import median
from skimage.morphology import disk

# --- your turn ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

show(speckled, title="salt & pepper", ax=axes[0][0])
show(...,  title="gaussian", ax=axes[0][1])   # TODO
show(...,  title="median",   ax=axes[0][2])   # TODO

show(grainy, title="grainy", ax=axes[1][0])
show(...,  title="gaussian", ax=axes[1][1])   # TODO
show(...,  title="median",   ax=axes[1][2])   # TODO

plt.tight_layout()
plt.show()

**Your answer:** … *(edit this cell)*

## Task 4: a full pipeline on the hard channel

Now the real one. Build a cleanup pipeline for `15min_3_cells.tif` and compare your
result against `15min_3_cells_labels.tif`.

Suggested steps, but you are free to deviate, and you should experiment:

1. smooth
2. threshold
3. fill holes
4. remove small objects

<details>
<summary>Hint 1: the imports you need</summary>

`from scipy.ndimage import binary_fill_holes` and
`from skimage.morphology import remove_small_objects`.
</details>

<details>
<summary>Hint 2: remove_small_objects has a new signature</summary>

In scikit-image 0.26 it is `remove_small_objects(mask, max_size=N)`, removing
objects smaller than **or equal to** `N`. Older examples online say `min_size`.
</details>

<details>
<summary>Hint 3: how small is too small?</summary>

Look at the object areas first, the way the walkthrough did:
`np.sort([(lab == i).sum() for i in range(1, lab.max() + 1)])`. Pick a cut below
the smallest believable cell.
</details>

In [ ]:
from scipy.ndimage import binary_fill_holes
from skimage.morphology import remove_small_objects

cells = tifffile.imread(DATA / "bbbc020" / "images" / "15min_3_cells.tif")
truth = tifffile.imread(DATA / "bbbc020" / "gt" / "15min_3_cells_labels.tif")

# --- your turn ---
naive = ...     # TODO: label the naively thresholded image, for comparison

smoothed = ...  # TODO: smooth
mask = ...      # TODO: threshold
mask = ...      # TODO: fill holes
mask = ...      # TODO: remove small objects
cleaned = label(mask)

print("naive:", naive.max(), " cleaned:", cleaned.max(), " truth:", truth.max())

**Your answer:** how close did you get, and what is still wrong with the result?

Look at the pictures, not just the number. Your count is probably *lower* than
the truth even though the binary image itself looks quite good: find a place in your
labeled image where that is happening and say what you see.
*(edit this cell)*

### Count and quality are different questions

Your mask probably overlaps the annotated cells rather well, and yet the count
came out too low. Both things are true at once, because several neighbouring
cells merged into a single label: one blob where the truth has two or three.

This is the trap in judging a segmentation by its object count: a count can be
close for the wrong reasons, or wrong while the segmentation is basically sound.
Day 2 introduces proper tools for asking *how good is this really*: 
per-object matching, and the IoU and Dice scores: instead of squinting at two
numbers.

## Task 5: split the merged cells with a watershed

Your count came out low because neighbouring cells merged. Section 6 of the
walkthrough splits touching objects with a watershed, so apply the same idea
here, but note that those parameters were tuned for **nuclei**, which are small
and round. These cells are bigger and much less regular, so you will have to
retune.

Fill in the three steps: distance transform, peaks, watershed.

<details>
<summary>Hint 1: the imports</summary>

```python
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
```
</details>

<details>
<summary>Hint 2: start with the simplest seeding</summary>

```python
distance = distance_transform_edt(mask)
coordinates = peak_local_max(distance, labels=mask)
```
Turn those coordinates into a marker image, then call
`watershed(-distance, markers, mask=mask)`. Expect it to over-split badly: that
is the point of trying it first.
</details>

<details>
<summary>Hint 3: building the marker image</summary>

`peak_local_max` returns coordinates, not an image. Number them:

```python
markers = np.zeros(mask.shape, dtype=int)
for i, (r, c) in enumerate(coordinates, start=1):
    markers[r, c] = i
```
</details>

<details>
<summary>Hint 4: then fix it</summary>

The walkthrough showed two ways: threshold the distance map (`distance > t`) so
only object cores become seeds, or smooth the distance map with a Gaussian and
find peaks on that. Try both. These cells are much larger than the nuclei in the
walkthrough, so any distance cut-off has to be much larger too.
</details>

In [ ]:
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import watershed
from skimage.feature import peak_local_max


def seeds_from_peaks(distance, mask):
    """Every local maximum of `distance` becomes its own seed."""
    coordinates = peak_local_max(distance, labels=mask)
    seeds = np.zeros(mask.shape, dtype=int)
    for i, (row, column) in enumerate(coordinates, start=1):
        seeds[row, column] = i
    return seeds


# --- your turn ---
distance = ...   # TODO: distance transform of the mask

# 1. the naive version - expect it to over-split
raw = ...        # TODO: watershed seeded from raw peaks
print(f"raw peaks -> {raw.max()} objects  (truth {truth.max()})")

# 2. seeds from a thresholded distance map
for t in (8, 12, 16, 20):
    print(...)   # TODO

# 3. seeds from peaks of a SMOOTHED distance map
for sigma in (2, 4, 6):
    print(...)   # TODO

The naive seeding over-splits badly, exactly as it did in the walkthrough. Both
remedies pull it back. Pick whichever gave you the most stable answer and look at
the result.

In [ ]:
smoothed = gaussian(distance, sigma=4, preserve_range=True)
split = watershed(-smoothed, seeds_from_peaks(smoothed, mask), mask=mask)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
show(cells, title="input image", ax=axes[0])
axes[1].imshow(label2rgb(cleaned, bg_label=0)); axes[1].set_axis_off()
axes[1].set_title(f"before watershed: {cleaned.max()}")
axes[2].imshow(label2rgb(split, bg_label=0)); axes[2].set_axis_off()
axes[2].set_title(f"after watershed: {split.max()}")
axes[3].imshow(label2rgb(truth, bg_label=0)); axes[3].set_axis_off()
axes[3].set_title(f"truth: {truth.max()}")
plt.show()

In [ ]:
# A blunt check you can run without any metrics: how many of your objects still
# swallow two or more annotated cells?
from skimage.measure import regionprops


def still_merged(prediction, truth):
    count = 0
    for region in regionprops(prediction):
        overlapping = set(np.unique(truth[prediction == region.label])) - {0}
        if len(overlapping) >= 2:
            count += 1
    return count


print("objects       :", split.max(), " (truth", truth.max(), ")")
print("still merged  :", still_merged(split, truth))

**Your answer:** compare the object count with the number of blobs that still
merge two or more annotated cells. Can the count look reasonable while the
segmentation is not? *(edit this cell)*

### Why the watershed struggles here

The watershed assumes each object has **one** summit in the distance map: a good
assumption for round nuclei and a poor one for these cells. An irregular, lumpy
cell has several local maxima, so it fragments; meanwhile two cells joined by a
broad bridge still look like a single hill and stay merged.

Both remedies trade one error against the other. Raising the distance cut-off, or
smoothing harder, suppresses spurious peaks, and eventually suppresses real ones
too. Somewhere in between is a compromise, not a solution, which is why an object
count near the truth here can easily be over-splits and merges cancelling out.

You have now spent half an hour hand-tuning a pipeline for **one channel of one
image**, and it still is not right. That is not a failure on your part: it is
the honest limit of writing the rules by hand. On day 2 the rules stop being written
and let a model learn them instead.